# Demo 3：AI 看圖認物

> Jupyter Notebook 版本 — 從「文字 AI」跳到「圖片 AI」，發現用法竟然一樣

## 這個 notebook 要做什麼？

你給 AI 一張圖片（網路上的或你電腦裡的），AI 告訴你「這張圖最有可能是什麼」。

這個任務叫 **Image Classification（圖像分類）**，是電腦視覺（Computer Vision）裡最基本的任務之一。

## 跟前兩個 Demo 比，最神奇的一件事

**你會發現用法幾乎一模一樣！**

| Demo | 程式碼 |
|---|---|
| Demo 1（文字） | `pipeline("sentiment-analysis")` |
| Demo 2（文字） | `pipeline("translation", model="...")` |
| Demo 3（圖片） | `pipeline("image-classification", model="...")` |

文字、圖片、聲音、影片……都可以用**同一個 `pipeline()`**。這就是 HuggingFace 設計的厲害之處：**抽象化**。

## 第一次跑會發生什麼？

- 「建立圖片分類機器」那格會下載 ViT 模型，約 350 MB，要等 1～2 分鐘
- 之後 AI 會分析一張範例圖片（一隻貓），印出前 5 名最有可能的答案

## 步驟 1：借工具（這個你已經看過 3 次了）

In [ ]:
from transformers import pipeline

## 步驟 2：建立圖片分類機器

- `"image-classification"` —— 任務名稱（看圖辨識物體）
- `model="google/vit-base-patch16-224"` —— Google 訓練的 **Vision Transformer (ViT)**

這個模型可以辨識 **1000 種**常見物體（動物、車、食物、家具等等）。

> ⏳ 第一次跑會下載模型（約 350 MB），耐心等。

In [ ]:
classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224",
)

## 步驟 3：準備一張圖

我們先用 HuggingFace 官方範例常用的測試圖（一隻坐在沙發上的貓）。

這張圖來自 COCO dataset，是電腦視覺界很有名的公開資料集。

In [ ]:
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"

### （加碼）先看一下這張圖長什麼樣

notebook 比 .py 強的地方之一：**可以直接顯示圖片**。

In [ ]:
from IPython.display import Image
Image(url=image_url, width=400)

## 步驟 4：AI 看圖

把圖片網址（或本地路徑）丟給 AI，請它告訴我們「前 5 名」最可能的答案。

`top_k=5` 意思是「給我前 5 名」，可以改成 `top_k=10` 看更多。

In [ ]:
results = classifier(image_url, top_k=5)
results

## 步驟 5：印出排行榜（用方塊畫信心度長條）

In [ ]:
print("=== AI 的猜測（信心度從高到低）===\n")
for rank, result in enumerate(results, start=1):
    label = result["label"]              # 物體名稱（英文）
    score = result["score"]              # 信心度 0～1
    bar = "█" * int(score * 30)          # 用方塊畫一個信心度長條
    print(f"  第 {rank} 名：{label}")
    print(f"          {bar} {score:.1%}\n")

## 觀察一下結果

第 1 名應該是「Egyptian cat（埃及貓）」之類的，信心 90% 上下。
其他幾名也全部都是貓的品種 —— AI 雖然不一定 100% 答對品種，但「**這是貓**」這件事 99% 確定。

這個現象叫做「**分布合理（reasonable distribution）**」 —— 即使猜錯第 1 名，前幾名也都在合理範圍內。你可以拿來判斷模型有沒有發瘋。

---

## 一個你應該知道的小細節：closed-set

這個模型只能辨識**事先定好的 1000 種**物體（叫做 ImageNet-1k 類別）。

如果你給它：
- 「你家狗狗」 → 它會告訴你品種，可能對也可能錯
- 「你的早餐拉麵」 → 它可能說 `soup bowl` 或 `pasta`
- 「某個 cosplay 人物」 → 它可能完全認不出，只說 `mask` 或 `costume`

**這個叫做 closed-set classification（封閉類別分類）**。

更厲害的模型（**CLIP**、BLIP）可以辨識「**任何你用文字描述的東西**」 —— 那叫 **open-vocabulary**。之後想學可以查 `CLIP zero-shot classification`。

## 動手玩玩看

### 玩法 1：換成你想辨識的網路圖片

把 `my_image_url` 換成任何網路圖片的網址（`.jpg` / `.png` 都行）。

找測試圖可以：
- Google Images 隨便抓一張的網址
- 用 https://picsum.photos/600/400 隨機圖

In [ ]:
my_image_url = "https://picsum.photos/600/400"  # ← 換成你想辨識的圖片網址

Image(url=my_image_url, width=400)

In [ ]:
for rank, result in enumerate(classifier(my_image_url, top_k=5), start=1):
    bar = "█" * int(result["score"] * 30)
    print(f"  第 {rank} 名：{result['label']}")
    print(f"          {bar} {result['score']:.1%}\n")

### 玩法 2：用你電腦裡的本地圖片

把網址換成本地路徑：

```python
# Windows 路徑前面要加 r 不然反斜線會出事
my_local_image = r"C:\Users\你的名字\Pictures\my_dog.jpg"

# Mac / Linux 用正斜線
# my_local_image = "/Users/你的名字/Pictures/my_dog.jpg"

for rank, result in enumerate(classifier(my_local_image, top_k=5), start=1):
    print(f"  第 {rank} 名：{result['label']} ({result['score']:.1%})")
```

### 玩法 3：批次辨識（一次跑很多張）

In [ ]:
image_urls = [
    "http://images.cocodataset.org/val2017/000000039769.jpg",  # 貓
    "https://picsum.photos/seed/dog/600/400",
    "https://picsum.photos/seed/food/600/400",
]

for url in image_urls:
    print(f"\n--- {url} ---")
    top3 = classifier(url, top_k=3)
    for rank, result in enumerate(top3, start=1):
        print(f"  第 {rank} 名：{result['label']} ({result['score']:.1%})")

## 你剛剛做了什麼？

**不誇張地說 —— 10 年前要博士才能做的事，你剛剛 8 行 Python 完成。**

你在這個系列學到了：

1. ✅ Demo 1：文字情感分析（`pipeline("sentiment-analysis")`）
2. ✅ Demo 2：英翻中（`pipeline("translation", model=...)`）
3. ✅ Demo 3：圖片分類（`pipeline("image-classification", model=...)`）

三種完全不同類型的 AI 任務，**同一個 `pipeline()` API**。

這就是現代 AI 工具的力量。歡迎入坑。

## 下一步

看 `docs/下一步.md`，裡面有：

- 高中生 / 大學生的學習路線圖
- 中期關鍵字（fine-tune、LoRA、RAG、Embedding）
- 接下來可以做的小專題